<a href="https://colab.research.google.com/github/tarunku/shruti_book/blob/main/colab_conversational_agent_langgraph.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🕉️ Production Conversational Agent — Colab Version
## LangGraph + Hybrid Search (ChromaDB + BM25)

**Prerequisites:** Run `colab_production_rag_pipeline.ipynb` first to ingest data.

**Architecture:**
- **Vector Search:** ChromaDB (loaded from Google Drive)
- **BM25 Search:** `rank_bm25` (loaded from pickle on Drive)
- **Fusion:** Reciprocal Rank Fusion (RRF)
- **Embedding:** `intfloat/multilingual-e5-large`
- **LLM:** AWS Bedrock (Nova Lite)

## Step 1: Mount Drive & Install

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("✅ Google Drive mounted")

Mounted at /content/drive
✅ Google Drive mounted


In [ ]:
!pip install -q chromadb rank_bm25 sentence-transformers langchain langchain-aws langchain-core langgraph boto3
print("\n✅ Packages installed — restart runtime if first time (Runtime → Restart session)")


✅ Packages installed — restart runtime if first time (Runtime → Restart session)


## Step 2: Imports

In [ ]:
import os
import json
import pickle
import numpy as np
from pathlib import Path
from typing import List, Dict, Any, TypedDict
from dataclasses import dataclass
from collections import defaultdict

import torch
from sentence_transformers import SentenceTransformer

import chromadb
from rank_bm25 import BM25Okapi

import boto3
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from langchain_aws import ChatBedrock
from langchain_core.messages import HumanMessage, SystemMessage

print("✅ Imports loaded")
print(f"   GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only'}")

✅ Imports loaded
   GPU: Tesla T4


## Step 3: Configuration

**Set your AWS credentials using Colab Secrets** (🔑 icon in left sidebar):
- `AWS_ACCESS_KEY_ID`
- `AWS_SECRET_ACCESS_KEY`
- `AWS_DEFAULT_REGION`

In [ ]:
from google.colab import userdata

# ─── AWS Credentials (from Colab Secrets) ─────────────────────────────────────
os.environ['AWS_ACCESS_KEY_ID']     = userdata.get('AWS_ACCESS_KEY_ID')
os.environ['AWS_SECRET_ACCESS_KEY'] = userdata.get('AWS_SECRET_ACCESS_KEY')
os.environ['AWS_DEFAULT_REGION']    = 'us-east-1' #userdata.get('AWS_DEFAULT_REGION', 'us-east-1')

AWS_REGION = os.environ['AWS_DEFAULT_REGION']

# ─── Embedding Model ───────────────────────────────────────────────────────────
EMBEDDING_MODEL_ID   = "intfloat/multilingual-e5-large"
EMBEDDING_MODEL_NAME = "multilingual-e5-large"
EMBEDDING_DIMENSION  = 1024

# ─── Google Drive Paths ────────────────────────────────────────────────────────
# ⚠️ Must match exactly what was set in the pipeline notebook
PROJECT_ROOT    = "/content/drive/MyDrive/ram_katha_rag"
CHROMA_DIR      = f"{PROJECT_ROOT}/chroma_db"
BM25_INDEX_PATH = f"{PROJECT_ROOT}/bm25_index.pkl"
COLLECTION_NAME = "discourse_chunks"

# ─── LLM ───────────────────────────────────────────────────────────────────────
# LLM_MODEL_ID = "us.amazon.nova-2-lite-v1:0"
LLM_MODEL_ID = "us.meta.llama3-2-90b-instruct-v1:0"
LLM_CONFIG   = {'temperature': 0.0, 'max_tokens': 8192}

# ─── Retrieval ─────────────────────────────────────────────────────────────────
RETRIEVAL_CONFIG = {
    'strategy'   : 'rrf',    # 'rrf', 'vector_only', 'bm25_only'
    'top_k'      : 10,
    'rrf_k'      : 60,
    'retrieval_k': 20
}

print("✅ Configuration loaded")
print(f"   Embedding model: {EMBEDDING_MODEL_ID}")
print(f"   LLM model      : {LLM_MODEL_ID}")
print(f"   Strategy       : {RETRIEVAL_CONFIG['strategy']}")
print(f"   ChromaDB path  : {CHROMA_DIR}")

✅ Configuration loaded
   Embedding model: intfloat/multilingual-e5-large
   LLM model      : us.meta.llama3-2-90b-instruct-v1:0
   Strategy       : rrf
   ChromaDB path  : /content/drive/MyDrive/ram_katha_rag/chroma_db


## Step 4: Data Classes

In [ ]:
@dataclass
class ChunkResult:
    """Retrieved chunk with metadata"""
    chunk_id        : str
    video_id        : str
    chunk_text      : str
    chunk_index     : int
    score           : float
    start_timestamp : str
    end_timestamp   : str
    contains_shloka : bool
    metadata        : Dict[str, Any]

print("✅ ChunkResult defined")

✅ ChunkResult defined


## Step 5: Embedding Generator

In [ ]:
class MultilingualE5EmbeddingGenerator:
    """
    intfloat/multilingual-e5-large
    IMPORTANT: 'query: ' prefix required at retrieval time.
    """
    MODEL_NAME = "intfloat/multilingual-e5-large"
    DIMENSION  = 1024

    def __init__(self):
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        print(f"Loading {self.MODEL_NAME} on {self.device}...")
        self.model = SentenceTransformer(self.MODEL_NAME, device=self.device)
        print(f"✅ {self.MODEL_NAME} loaded (dim={self.DIMENSION})")

    def invoke(self, text: str) -> List[float]:
        """Single embedding for query — adds 'query: ' prefix"""
        return self.model.encode(
            f"query: {text}",
            normalize_embeddings=True
        ).tolist()

print("✅ MultilingualE5EmbeddingGenerator defined")

✅ MultilingualE5EmbeddingGenerator defined


## Step 6: Retrieval Components

In [ ]:
class VectorRetriever:
    """Vector similarity search using ChromaDB"""

    def __init__(self, embedding_gen: MultilingualE5EmbeddingGenerator):
        self.embedding_gen = embedding_gen
        self.client        = chromadb.PersistentClient(path=CHROMA_DIR)
        self.collection    = self.client.get_collection(COLLECTION_NAME)
        print(f"✅ VectorRetriever ready — {self.collection.count()} documents in collection")

    def search(self, query: str, k: int = 10) -> List[ChunkResult]:
        query_embedding = self.embedding_gen.invoke(query)

        results = self.collection.query(
            query_embeddings=[query_embedding],
            n_results=k,
            include=['documents', 'metadatas', 'distances']
        )

        chunks = []
        for doc, meta, dist in zip(
            results['documents'][0],
            results['metadatas'][0],
            results['distances'][0]
        ):
            # Chroma returns cosine distance (0=identical, 2=opposite)
            # Convert to similarity score for consistency
            similarity = 1 - dist
            chunks.append(ChunkResult(
                chunk_id        = results['ids'][0][len(chunks)],
                video_id        = meta.get('video_id', ''),
                chunk_text      = doc,
                chunk_index     = meta.get('chunk_index', 0),
                score           = similarity,
                start_timestamp = meta.get('start_timestamp', ''),
                end_timestamp   = meta.get('end_timestamp', ''),
                contains_shloka = meta.get('contains_shloka', False),
                metadata        = meta
            ))
        return chunks

print("✅ VectorRetriever defined")

✅ VectorRetriever defined


In [ ]:
class BM25Retriever:
    """
    BM25 keyword search using rank_bm25.
    Loads pre-built index from Google Drive pickle.
    """

    def __init__(self):
        with open(BM25_INDEX_PATH, 'rb') as f:
            data = pickle.load(f)

        self.bm25      = data['bm25']
        self.chunk_ids = data['chunk_ids']
        self.texts     = data['texts']
        self.metadatas = data['metadatas']
        print(f"✅ BM25Retriever ready — {len(self.chunk_ids)} documents in index")

    def search(self, query: str, k: int = 10) -> List[ChunkResult]:
        tokenized_query = query.split()
        scores          = self.bm25.get_scores(tokenized_query)
        top_indices     = np.argsort(scores)[::-1][:k]

        chunks = []
        for idx in top_indices:
            if scores[idx] == 0:
                continue   # skip zero-score results
            meta = self.metadatas[idx]
            chunks.append(ChunkResult(
                chunk_id        = self.chunk_ids[idx],
                video_id        = meta.get('video_id', ''),
                chunk_text      = self.texts[idx],
                chunk_index     = meta.get('chunk_index', 0),
                score           = float(scores[idx]),
                start_timestamp = meta.get('start_timestamp', ''),
                end_timestamp   = meta.get('end_timestamp', ''),
                contains_shloka = meta.get('contains_shloka', False),
                metadata        = meta
            ))
        return chunks

print("✅ BM25Retriever defined")

✅ BM25Retriever defined


In [ ]:
def reciprocal_rank_fusion(
    results_lists: List[List[ChunkResult]],
    k: int = 60
) -> List[ChunkResult]:
    """Combine multiple result lists using Reciprocal Rank Fusion"""
    rrf_scores = defaultdict(float)
    chunk_map  = {}

    for results in results_lists:
        for rank, chunk in enumerate(results, start=1):
            rrf_scores[chunk.chunk_id] += 1.0 / (k + rank)
            if chunk.chunk_id not in chunk_map:
                chunk_map[chunk.chunk_id] = chunk

    sorted_chunks = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)

    merged = []
    for chunk_id, rrf_score in sorted_chunks:
        chunk       = chunk_map[chunk_id]
        chunk.score = rrf_score
        merged.append(chunk)
    return merged

print("✅ Reciprocal Rank Fusion defined")

✅ Reciprocal Rank Fusion defined


In [ ]:
class HybridRetriever:
    """Hybrid retrieval combining Vector + BM25 with RRF"""

    def __init__(self, embedding_gen: MultilingualE5EmbeddingGenerator):
        self.vector_retriever = VectorRetriever(embedding_gen)
        self.bm25_retriever   = BM25Retriever()

    def search(
        self,
        query      : str,
        k          : int = 10,
        strategy   : str = 'rrf',
        rrf_k      : int = 60,
        retrieval_k: int = None
    ) -> List[ChunkResult]:
        if retrieval_k is None:
            retrieval_k = k * 2 if strategy == 'rrf' else k

        if strategy == 'vector_only':
            return self.vector_retriever.search(query, k)

        elif strategy == 'bm25_only':
            return self.bm25_retriever.search(query, k)

        elif strategy == 'rrf':
            vector_results = self.vector_retriever.search(query, retrieval_k)
            bm25_results   = self.bm25_retriever.search(query, retrieval_k)
            fused          = reciprocal_rank_fusion([vector_results, bm25_results], k=rrf_k)
            return fused[:k]

        else:
            raise ValueError(f"Unknown strategy: {strategy}")

print("✅ HybridRetriever defined")

✅ HybridRetriever defined


## Step 7: LLM Client

In [ ]:
def get_llm(temperature: float = 0.0) -> ChatBedrock:
    return ChatBedrock(
        model_id     = LLM_MODEL_ID,
        model_kwargs = {'temperature': temperature, 'max_tokens': LLM_CONFIG['max_tokens']},
        region_name  = AWS_REGION
    )

print("✅ LLM helper defined")

✅ LLM helper defined


## Step 8: System Prompt

In [ ]:
SYSTEM_PROMPT = """You are a knowledgeable assistant specialized in answering questions about spiritual discourse, particularly focusing on Hindu/Sanskrit religious teachings from the Ramayana and related texts.

CRITICAL INSTRUCTIONS - FOLLOW STRICTLY:

1. **Answer ONLY based on the provided context**
   - You will be given relevant excerpts from spiritual discourse transcripts, each labelled with a [Cref-N] identifier and a timestamp
   - Use ONLY information present in these excerpts
   - Do NOT use any external knowledge or information not in the context

2. **If the answer is NOT in the context:**
   - Say: "इस प्रश्न का उत्तर प्रदान किए गए प्रवचन में उपलब्ध नहीं है।"
   - Do NOT attempt to answer from general knowledge
   - Do NOT make up or infer information

3. **Response format — DETAILED AND CONTEXTUAL:**
   - Answer in Hindi (same language as the context)
   - Every answer must go beyond a single-line reply. Provide the necessary background, the core teaching being discussed, and the larger spiritual significance — all drawn from the context
   - Integrate surrounding context naturally so the answer reads as a coherent, flowing explanation rather than a bare fact followed by appended notes
   - Quote or closely paraphrase Sanskrit shlokas or key phrases where they enrich understanding, preserving them exactly as they appear
   - Inline citations: after each claim or passage drawn from a specific excerpt, append the reference tag in the form [Cref-N] immediately at the end of that sentence

4. **Citations block — MANDATORY:**
   - After the main answer, append a citations block with the exact heading: "---\n## CITATIONS"
   - List every [Cref-N] reference used, one per line, in this exact machine-parsable format:
     [Cref-N] | <chunk_id> | <start_timestamp> - <end_timestamp>
   - Only include references that were actually cited in the answer body
   - Do not add any prose or explanation inside the citations block

5. **Content handling:**
   - Preserve Sanskrit shlokas exactly as they appear
   - Maintain respectful, reverential tone appropriate for spiritual content

6. **Verification:**
   - Before answering, verify each claim exists in the provided context
   - If uncertain, err on the side of saying the information is not available

Remember: Your credibility depends on NEVER hallucinating and ALWAYS providing traceable citations.
"""

print("✅ System prompt defined")

✅ System prompt defined


## Step 9: LangGraph Agent

In [ ]:
class AgentState(TypedDict):
    query               : str
    retrieved_chunks    : List[ChunkResult]
    formatted_context   : str
    response            : str
    retrieval_strategy  : str
    num_chunks_retrieved: int

print("✅ AgentState defined")

✅ AgentState defined


In [ ]:
class ConversationalAgent:
    """LangGraph-based conversational agent with hybrid retrieval"""

    def __init__(
        self,
        retriever       : HybridRetriever,
        llm             : ChatBedrock,
        retrieval_config: Dict
    ):
        self.retriever        = retriever
        self.llm              = llm
        self.retrieval_config = retrieval_config
        self.graph            = self._build_graph()

    def _build_graph(self) -> StateGraph:
        workflow = StateGraph(AgentState)
        workflow.add_node("retrieve",       self._retrieve_node)
        workflow.add_node("format_context", self._format_context_node)
        workflow.add_node("generate",       self._generate_node)
        workflow.set_entry_point("retrieve")
        workflow.add_edge("retrieve",       "format_context")
        workflow.add_edge("format_context", "generate")
        workflow.add_edge("generate",       END)
        memory = MemorySaver()
        return workflow.compile(checkpointer=memory)

    def _retrieve_node(self, state: AgentState) -> AgentState:
        chunks = self.retriever.search(
            query       = state['query'],
            k           = self.retrieval_config['top_k'],
            strategy    = self.retrieval_config['strategy'],
            rrf_k       = self.retrieval_config['rrf_k'],
            retrieval_k = self.retrieval_config['retrieval_k']
        )
        return {**state, 'retrieved_chunks': chunks,
                'retrieval_strategy': self.retrieval_config['strategy'],
                'num_chunks_retrieved': len(chunks)}

    def _format_context_node(self, state: AgentState) -> AgentState:
        chunks = state['retrieved_chunks']
        if not chunks:
            formatted = "कोई संदर्भ नहीं मिला। (No context found.)"
        else:
            parts = [
                (
                    f"[Cref-{i}]\n"
                    f"chunk_id : {c.chunk_id}\n"
                    f"समय     : {c.start_timestamp} - {c.end_timestamp}\n\n"
                    f"{c.chunk_text}"
                )
                for i, c in enumerate(chunks, 1)
            ]
            formatted = "\n\n---\n\n".join(parts)
        return {**state, 'formatted_context': formatted}

    def _generate_node(self, state: AgentState) -> AgentState:
        messages = [
            SystemMessage(content=SYSTEM_PROMPT),
            HumanMessage(content=f"""प्रवचन संदर्भ (Discourse Context):

{state['formatted_context']}

---

प्रश्न (Question): {state['query']}

निर्देश:
1. केवल ऊपर दिए गए संदर्भों ([Cref-1], [Cref-2], …) के आधार पर उत्तर दें।
2. उत्तर विस्तृत और संदर्भ-सहित हो — केवल एक पंक्ति में उत्तर न दें। प्रासंगिक पृष्ठभूमि, मुख्य शिक्षा और आध्यात्मिक महत्व को प्राकृतिक रूप से उत्तर में सम्मिलित करें।
3. प्रत्येक तथ्य के बाद उसका सन्दर्भ [Cref-N] inline अंकित करें।
4. उत्तर के अंत में "---\\n## CITATIONS" शीर्षक के नीचे उपयोग किए गए सभी [Cref-N] को निर्धारित प्रारूप में सूचीबद्ध करें।
""")
        ]
        response = self.llm.invoke(messages)
        return {**state, 'response': response.content}

    def query(self, question: str, thread_id: str = "default") -> Dict[str, Any]:
        initial_state = {
            'query'               : question,
            'retrieved_chunks'    : [],
            'formatted_context'   : '',
            'response'            : '',
            'retrieval_strategy'  : '',
            'num_chunks_retrieved': 0
        }
        config      = {"configurable": {"thread_id": thread_id}}
        final_state = self.graph.invoke(initial_state, config)
        return {
            'question'           : question,
            'answer'             : final_state['response'],
            'retrieval_strategy' : final_state['retrieval_strategy'],
            'num_chunks'         : final_state['num_chunks_retrieved'],
            'chunks'             : final_state['retrieved_chunks']
        }

print("✅ ConversationalAgent defined")

✅ ConversationalAgent defined


## Step 10: Initialize Agent

In [ ]:
print("Initializing agent components...\n")

embedding_gen = MultilingualE5EmbeddingGenerator()
retriever     = HybridRetriever(embedding_gen)
llm           = get_llm(temperature=LLM_CONFIG['temperature'])
agent         = ConversationalAgent(retriever, llm, RETRIEVAL_CONFIG)

print(f"\n✅ Agent ready")
print(f"   Strategy: {RETRIEVAL_CONFIG['strategy']}")
print(f"   LLM     : {LLM_MODEL_ID}")

Initializing agent components...

Loading intfloat/multilingual-e5-large on cuda...


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

✅ intfloat/multilingual-e5-large loaded (dim=1024)
✅ VectorRetriever ready — 772 documents in collection
✅ BM25Retriever ready — 599 documents in index

✅ Agent ready
   Strategy: rrf
   LLM     : us.meta.llama3-2-90b-instruct-v1:0


## Step 11: Test Queries

In [ ]:
test_queries = [
    "समय का सार्थक उपयोग क्या है?",
    "हम भक्ति क्यों करते हैं?",
    "वेदों का मुख्य उद्देश्य क्या है?",
    "रामायण के रचयिता कौन हैं?",
    "शरणागति क्या है?",
]
print(f"✅ {len(test_queries)} test queries ready")

✅ 5 test queries ready


In [ ]:
# Run a single test query — change index to try different questions
query  = test_queries[0]
result = agent.query(query)

print(f"📍 Question: {query}\n")
print(f"💬 Answer:\n{result['answer']}")
# print(f"\n📊 Metadata:")
# print(f"   Strategy: {result['retrieval_strategy']} | Chunks: {result['num_chunks']}")
# print(f"\n📚 Top 3 Sources:")
# for i, chunk in enumerate(result['chunks'][:3], 1):
#     print(f"   [{i}] {chunk.start_timestamp} - {chunk.end_timestamp} | Score: {chunk.score:.4f}")
#     print(f"       {chunk.chunk_text[:200]}...\n")

📍 Question: समय का सार्थक उपयोग क्या है?

💬 Answer:
समय का सार्थक उपयोग एक ऐसा विषय है जिस पर गोस्वामी जी महाराज ने अपने प्रवचन में विस्तार से चर्चा की है [Cref-4]। उनके अनुसार, समय का सार्थक उपयोग करने के लिए हमें सत्संग में समय बिताना चाहिए। सत्संग से हमें जीवन के सही मार्ग की प्राप्ति होती है और हमारे जीवन में सकारात्मक परिवर्तन आते हैं [Cref-2]। गोस्वामी जी महाराज कहते हैं कि सत्संग में समय बिताने से हमारे जीवन का उद्देश्य स्पष्ट होता है और हम अपने जीवन को सही दिशा में आगे बढ़ा पाते हैं [Cref-4]।

सत्संग के महत्व को समझाने के लिए गोस्वामी जी महाराज ने एक उदाहरण दिया है [Cref-4]। उन्होंने कहा है कि जैसे एक नाविक अपनी नाव को सुरक्षित रूप से नदी पार कराने के लिए अनुभवी नाविक की सलाह लेता है, वैसे ही हमें भी अपने जीवन को सार्थक बनाने के लिए सत्संग में समय बिताना चाहिए। सत्संग से हमें जीवन के सही मार्ग की प्राप्ति होती है और हम अपने जीवन को सही दिशा में आगे बढ़ा पाते हैं [Cref-4]।

इसके अलावा, गोस्वामी जी महाराज ने यह भी कहा है कि समय का सार्थक उपयोग करने के लिए हमें भगवान की कृपा की आव

In [ ]:
# Run all test queries
for query in test_queries:
    result = agent.query(query)
    print(f"{'='*70}")
    print(f"📍 {query}")
    print(f"💬 {result['answer']}...")
    # print(f"   Chunks: {result['num_chunks']} | Strategy: {result['retrieval_strategy']}\n")

📍 समय का सार्थक उपयोग क्या है?
💬 समय का सार्थक उपयोग एक ऐसा विषय है जिस पर गोस्वामी जी महाराज ने अपने प्रवचन में विशेष रूप से प्रकाश डाला है। उनके अनुसार, समय का सार्थक उपयोग करने के लिए हमें सत्संग में समय बिताना चाहिए। सत्संग से हमें जीवन के सही मार्ग की प्राप्ति होती है और हमारे जीवन में सकारात्मक परिवर्तन आते हैं [Cref-2]।

गोस्वामी जी महाराज कहते हैं कि सत्संग में समय बिताने से हमारे जीवन का उद्देश्य स्पष्ट होता है और हमें अपने जीवन को सही दिशा में आगे बढ़ाने की प्रेरणा मिलती है। वे कहते हैं कि सत्संग में समय बिताने से हमारे जीवन में आनंद और शांति की प्राप्ति होती है और हम अपने जीवन को सार्थक बना पाते हैं [Cref-2]।

इसके अलावा, गोस्वामी जी महाराज यह भी कहते हैं कि समय का सार्थक उपयोग करने के लिए हमें अपने जीवन में परमार्थ के कार्यों को करना चाहिए। परमार्थ के कार्यों से हमारे जीवन में सकारात्मक परिवर्तन आते हैं और हम अपने जीवन को सार्थक बना पाते हैं [Cref-9]।

इस प्रकार, समय का सार्थक उपयोग करने के लिए हमें सत्संग में समय बिताना चाहिए और अपने जीवन में परमार्थ के कार्यों को करना चाह

In [ ]:
# Hallucination prevention test (out-of-scope query)
result = agent.query("आज का मौसम कैसा है?")
print(f"📍 Out-of-scope: 'आज का मौसम कैसा है?'\n")
print(f"💬 Answer:\n{result['answer']}")
print("\n✅ Agent should refuse or say information not available")

📍 Out-of-scope: 'आज का मौसम कैसा है?'

💬 Answer:
आज का मौसम कैसा है, इसके बारे में संदर्भों में कोई स्पष्ट जानकारी नहीं दी गई है। हालांकि, [Cref-4] में एक प्रसंग आता है जहां बारिश और बिजली की चर्चा होती है, लेकिन यह आज के मौसम की स्थिति के बारे में नहीं है। इसलिए, इस प्रश्न का उत्तर प्रदान किए गए प्रवचन में उपलब्ध नहीं है।

---
## CITATIONS
[Cref-4] | 🔴 LIVE !! DAY-05 !! श्री राम कथा जामुल भिलाई छत्तीसगढ़ !!  स्वामी श्री राघवाचार्य जी महाराज_chunks_cleaned_chunk_0027 | 01:08:40.560 - 01:10:50.470

✅ Agent should refuse or say information not available


## Step 12: Interactive Interface

In [ ]:
def ask_agent(question: str, show_sources: bool = True):
    """
    Interactive interface. Usage: ask_agent('आपका प्रश्न यहाँ')
    """
    print(f"\n{'='*70}")
    print(f"📍 प्रश्न: {question}")
    print(f"{'='*70}\n")

    result = agent.query(question)

    # print(f"💬 उत्तर:\n{result['answer']}")
    # print(f"\n📊 रणनीति: {result['retrieval_strategy']} | खंड: {result['num_chunks']}")

    # if show_sources and result['chunks']:
    #     print(f"\n📚 स्रोत (Top 3):")
    #     for i, chunk in enumerate(result['chunks'][:3], 1):
    #         print(f"\n  [{i}] {chunk.start_timestamp} - {chunk.end_timestamp} | Score: {chunk.score:.4f}")
    #         print(f"      {chunk.chunk_text[:200]}...")

    # print(f"\n{'='*70}\n")
    return result

# print("✅ Interactive interface ready")
# print("Usage: ask_agent('आपका प्रश्न यहाँ')")

In [ ]:

# ask_agent("पाप का अंत कैसे हो सकता है?")
# result = ask_agent("क्या जनक जी को ब्रम्ह का ज्ञान था ? यदि था थो वो भगवन को क्यों नहीं पहचान पाए? ")
# ask_agent("परशुराम जी और भगवन राम के मध्य क्या संवाद हुआ?")
# ask_agent("माता सीता ने भगवन राम को कैसे प्राप्त किया?")
# ask_agent("भगवान के गुणों का चिंतन कैसे करे? कुछ प्रमुख भगवत गुंणों के विषय में बताएं ")
# ask_agent("रामायण के बिभिन्न पात्रो ने भगवन के गुंणों का चिंतन और वर्णन कैसे किया?")
# ask_agent("सीता जी का लंका जाने का क्या प्रयोजन था?")
result = ask_agent("धर्म को कैसे पारिभाषित करे  तथा  धर्म के प्रकार क्या है?")
print(f"💬 उत्तर:\n{result['answer']}")


📍 प्रश्न: धर्म को कैसे पारिभाषित करे  तथा  धर्म के प्रकार क्या है?

💬 उत्तर:
धर्म की परिभाषा और प्रकार को समझने के लिए, हमें सबसे पहले यह जानना होगा कि धर्म क्या है। धर्म को एक व्यापक और जटिल अवधारणा के रूप में परिभाषित किया जा सकता है, जो व्यक्ति के जीवन को निर्देशित करने वाले नैतिक और आध्यात्मिक सिद्धांतों का समूह है। [Cref-1]

धर्म के प्रकारों की बात करते हुए, हमें यह जानना होगा कि धर्म को मुख्य रूप से दो श्रेणियों में विभाजित किया जा सकता है: सामान्य धर्म और विशेष धर्म। सामान्य धर्म वह है जो सभी व्यक्तियों पर लागू होता है, जबकि विशेष धर्म विशिष्ट वर्गों या समुदायों के लिए होता है। [Cref-4]

सामान्य धर्म के अंतर्गत, हमें सत्य बोलना, माता-पिता की सेवा करना, और अपने कर्तव्यों का पालन करना शामिल है। [Cref-4] यह धर्म सभी व्यक्तियों के लिए समान रूप से लागू होता है, चाहे वे किसी भी वर्ग या समुदाय से संबंधित हों।

विशेष धर्म, दूसरी ओर, विशिष्ट वर्गों या समुदायों के लिए होता है। उदाहरण के लिए, क्षत्रियों के लिए, उनका धर्म मृगया (शिकार) करना हो सकता है, जबकि ब्राह्मणों के लिए, उनका धर्म वेद

In [ ]:
result['chunks'][0]

ChunkResult(chunk_id='🔴 LIVE !! DAY-08 !! श्री राम कथा जामुल भिलाई छत्तीसगढ़ !!  स्वामी श्री राघवाचार्य जी महाराज_chunks_cleaned_chunk_0032', video_id='🔴 LIVE !! DAY-08 !! श्री राम कथा जामुल भिलाई छत्तीसगढ़ !!  स्वामी श्री राघवाचार्य जी महाराज_chunks_cleaned', chunk_text='[संगीत] वही हमारी रुचि राजी है हम उसी में जिसमें तेरी रजा है यह भरत जी का विशेष धर्म विशेष तर धर्म और इस विशेष तर धर्म से भी ऊंचा धर्म शत्रुघ्न जी का है सबसे छोटे भाई लेकिन इनका धर्म विशेषतम धर्म विशेषतम धर्म क्या होता है? बोले उसके आगे कुछ नहीं होता। फिर सबसे बड़ा धर्म क्योंकि संस्कृत में अधिकता को बताने के लिए जो शब्द बनता है प्रत्यय होता है। तर तम तम से आगे कुछ नहीं होता। जैसे अधिक और अधिक से भी ज्यादा हो तो क्या कहेंगे? बोले अधिकतर बोले अधिकतर और उससे भी ज्यादा हो तो क्या कहेंगे? बोले अधिकतम फिर उसके आगे कुछ नहीं। अधिकतम माने सबसे ज्यादा तो तम प्रत्यय जहां पर लग जाता है वह परिपूर्णता का बोध कराता है। तो शत्रुघ्न जी का जो धर्म है वह विशेषतम धर्म। विशेष तर्म धर्म क्या है? बोले इसमें विशेष तो जुड़ा है लेकिन विशेष के 

In [ ]:
# ── Type your question here ──────────────────────────────────────────────────
# ask_agent("सत्संग का क्या महत्व है?")

## Step 13: Debug Cells

In [ ]:
def debug_vector_search(query_text: str, top_k: int = 10):
    """Debug ChromaDB vector search results with similarity scores"""
    eg         = MultilingualE5EmbeddingGenerator()
    q_emb      = eg.invoke(query_text)
    collection = chromadb.PersistentClient(path=CHROMA_DIR).get_collection(COLLECTION_NAME)

    results = collection.query(
        query_embeddings=[q_emb],
        n_results=top_k,
        include=['documents', 'metadatas', 'distances']
    )

    print(f"🔍 Vector Search: '{query_text}'\n")
    print(f"{'='*80}")
    for i, (doc_id, doc, meta, dist) in enumerate(
        zip(results['ids'][0], results['documents'][0],
            results['metadatas'][0], results['distances'][0]), 1
    ):
        sim = 1 - dist
        print(f"[{i}] {doc_id} | Distance: {dist:.4f} | Similarity: {sim:.4f}")
        print(f"    Time: {meta['start_timestamp']} - {meta['end_timestamp']}")
        print(f"    {doc[:120]}...")
        print(f"    {'-'*76}")


debug_vector_search("समय का सार्थक उपयोग क्या है?")

Loading intfloat/multilingual-e5-large on cuda...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-large
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ intfloat/multilingual-e5-large loaded (dim=1024)
🔍 Vector Search: 'समय का सार्थक उपयोग क्या है?'

[1] 🔴D- LIVE !! DAY-01 !! श्री राम कथा जामुल भिलाई छत्तीसगढ़ !!  स्वामी श्री राघवाचार्य जी महाराज_chunks_cleaned_chunk_0036 | Distance: 0.1844 | Similarity: 0.8156
    Time: 01:29:30.239 - 01:31:50.870
    चर्चा श्रवण करेंगे जिससे हमारे जीवन में बहुत कुछ प्रकाश होगा और मार्ग मिलेगा और जितना भी हम समय भगवान की चर्चा में बिता ...
    ----------------------------------------------------------------------------
[2] 🔴D- LIVE !! DAY-02 !! श्री राम कथा जामुल भिलाई छत्तीसगढ़ !!  स्वामी श्री राघवाचार्य जी महाराज_chunks_cleaned_chunk_0009 | Distance: 0.1990 | Similarity: 0.8010
    Time: 00:27:18.159 - 00:29:17.909
    यह हमसे छूटती जाती है। हम ना समय को रोक पाते हैं ना काल के क्रम को रोक पाते हैं। जो कुछ हमें संसार में मिलता है वह सब धी...
    ----------------------------------------------------------------------------
[3] 🔴D- LIVE !! DAY-02 !! श्री राम कथा जामुल भिलाई छत्तीसगढ़ !!  स्वामी श्री र

In [ ]:
def debug_bm25_search(query_text: str, top_k: int = 10):
    """Debug BM25 search results with scores"""
    with open(BM25_INDEX_PATH, 'rb') as f:
        data = pickle.load(f)

    tokenized_query = query_text.split()
    scores          = data['bm25'].get_scores(tokenized_query)
    top_indices     = np.argsort(scores)[::-1][:top_k]

    print(f"🔍 BM25 Search: '{query_text}'\n")
    print(f"{'='*80}")

    found = False
    for i, idx in enumerate(top_indices, 1):
        if scores[idx] == 0:
            break
        found = True
        meta = data['metadatas'][idx]
        print(f"[{i}] {data['chunk_ids'][idx]} | BM25 Score: {scores[idx]:.4f}")
        print(f"    Time: {meta['start_timestamp']} - {meta['end_timestamp']}")
        print(f"    {data['texts'][idx][:120]}...")
        print(f"    {'-'*76}")

    if not found:
        print("⚠️ NO RESULTS — query tokens not found in any document")
        print(f"   Tokens searched: {tokenized_query}")


debug_bm25_search("समय सार्थक")

🔍 BM25 Search: 'समय सार्थक'

[1] 🔴D- LIVE !! DAY-01 !! श्री राम कथा जामुल भिलाई छत्तीसगढ़ !!  स्वामी श्री राघवाचार्य जी महाराज_chunks_cleaned_chunk_0036 | BM25 Score: 7.3668
    Time: 01:29:30.239 - 01:31:50.870
    चर्चा श्रवण करेंगे जिससे हमारे जीवन में बहुत कुछ प्रकाश होगा और मार्ग मिलेगा और जितना भी हम समय भगवान की चर्चा में बिता ...
    ----------------------------------------------------------------------------
[2] 🔴D- LIVE !! DAY-02 !! श्री राम कथा जामुल भिलाई छत्तीसगढ़ !!  स्वामी श्री राघवाचार्य जी महाराज_chunks_cleaned_chunk_0027 | BM25 Score: 2.5182
    Time: 01:03:51.359 - 01:05:43.829
    जब उस आयु के भोग का समय आया तब राम जी ने विचार किया कि मैं पिता की आयु का भोग करूं करूं और उस समय जब मैं पिता की आयु का ...
    ----------------------------------------------------------------------------
[3] 🔴D- LIVE !! DAY-01 !! श्री राम कथा जामुल भिलाई छत्तीसगढ़ !!  स्वामी श्री राघवाचार्य जी महाराज_chunks_cleaned_chunk_0062 | BM25 Score: 2.4518
    Time: 02:17:16.399 - 02:18:42.950
    

In [ ]:
def debug_hybrid_search(query_text: str, top_k: int = 10):
    """Debug full RRF fusion — shows vector rank, BM25 rank, and final RRF score"""
    retrieval_k    = top_k * 2
    vector_results = retriever.vector_retriever.search(query_text, retrieval_k)
    bm25_results   = retriever.bm25_retriever.search(query_text, retrieval_k)
    fused_results  = reciprocal_rank_fusion([vector_results, bm25_results], k=60)

    # Build rank maps
    v_ranks = {c.chunk_id: i for i, c in enumerate(vector_results, 1)}
    b_ranks = {c.chunk_id: i for i, c in enumerate(bm25_results, 1)}

    print(f"🔍 Hybrid RRF Search: '{query_text}'\n")
    print(f"{'='*90}")
    print(f"{'Rank':<5} {'Chunk ID':<30} {'Vec Rank':<10} {'BM25 Rank':<11} {'RRF Score':<12} Time")
    print(f"{'='*90}")

    for i, chunk in enumerate(fused_results[:top_k], 1):
        v_rank = v_ranks.get(chunk.chunk_id, '-')
        b_rank = b_ranks.get(chunk.chunk_id, '-')
        print(f"{i:<5} {chunk.chunk_id:<30} {str(v_rank):<10} {str(b_rank):<11} {chunk.score:.6f}   {chunk.start_timestamp}")


debug_hybrid_search("समय का सार्थक उपयोग क्या है?")

🔍 Hybrid RRF Search: 'समय का सार्थक उपयोग क्या है?'

Rank  Chunk ID                       Vec Rank   BM25 Rank   RRF Score    Time
1     🔴D- LIVE !! DAY-01 !! श्री राम कथा जामुल भिलाई छत्तीसगढ़ !!  स्वामी श्री राघवाचार्य जी महाराज_chunks_cleaned_chunk_0036 1          1           0.032787   01:29:30.239
2     🔴D- LIVE !! DAY-01 !! श्री राम कथा जामुल भिलाई छत्तीसगढ़ !!  स्वामी श्री राघवाचार्य जी महाराज_chunks_cleaned_chunk_0041 6          3           0.031025   01:39:32.239
3     🔴D- LIVE !! DAY-01 !! श्री राम कथा जामुल भिलाई छत्तीसगढ़ !!  स्वामी श्री राघवाचार्य जी महाराज_chunks_cleaned_chunk_0042 10         2           0.030415   01:41:05.520
4     🔴D- LIVE !! DAY-01 !! श्री राम कथा जामुल भिलाई छत्तीसगढ़ !!  स्वामी श्री राघवाचार्य जी महाराज_chunks_cleaned_chunk_0057 13         6           0.028850   02:08:09.788
5     🔴D- LIVE !! DAY-02 !! श्री राम कथा जामुल भिलाई छत्तीसगढ़ !!  स्वामी श्री राघवाचार्य जी महाराज_chunks_cleaned_chunk_0077 4          20          0.028125   02:35:21.280
6   

In [ ]:
def analyze_collection():
    """Show collection stats and sample documents"""
    client     = chromadb.PersistentClient(path=CHROMA_DIR)
    collection = client.get_collection(COLLECTION_NAME)

    count = collection.count()
    print(f"📊 Collection: '{COLLECTION_NAME}'")
    print(f"   Total documents: {count}")

    if count > 0:
        sample = collection.get(limit=3, include=['documents', 'metadatas'])
        print(f"\n   Sample documents:")
        for doc_id, doc, meta in zip(sample['ids'], sample['documents'], sample['metadatas']):
            print(f"\n   ID: {doc_id}")
            print(f"   Model: {meta.get('embedding_model')} | Time: {meta.get('start_timestamp')} - {meta.get('end_timestamp')}")
            print(f"   Text: {doc[:100]}...")


analyze_collection()

📊 Collection: 'discourse_chunks'
   Total documents: 173

   Sample documents:

   ID: 🔴D- LIVE !! DAY-01 !! श्री राम कथा जामुल भिलाई छत्तीसगढ़ !!  स्वामी श्री राघवाचार्य जी महाराज_chunks_cleaned_chunk_0000
   Model: multilingual-e5-large | Time: 00:00:00.400 - 00:02:04.789
   Text: परम पूज्य श्रीमद् जगतगुरु रामानुजाचार्य श्री स्वामी डॉ राघवाचार्य जी महाराज श्रीधाम मठ का प्राकट फाल...

   ID: 🔴D- LIVE !! DAY-01 !! श्री राम कथा जामुल भिलाई छत्तीसगढ़ !!  स्वामी श्री राघवाचार्य जी महाराज_chunks_cleaned_chunk_0001
   Model: multilingual-e5-large | Time: 00:02:04.799 - 00:04:33.270
   Text: एक-एक बच्चे ने जिस तरीके से साथ निभाया है, आसपास पूरे क्षेत्र में जिस तरीके से इस आयोजन को सफल और सं...

   ID: 🔴D- LIVE !! DAY-01 !! श्री राम कथा जामुल भिलाई छत्तीसगढ़ !!  स्वामी श्री राघवाचार्य जी महाराज_chunks_cleaned_chunk_0002
   Model: multilingual-e5-large | Time: 00:04:33.280 - 00:08:55.190
   Text: रामम रघुवंशनाथम ध्यान अक्षत पुष्पण समर्पयाम श्री सीत रामचंद्राभ्याम रामचंद्राभ्याम नम हरि ओम गंधार